# Wildfire GRPO Training (Hugging Face)

Use this notebook only for GRPO training on the Hugging Face GPU runtime.

Cell order:
1. Run Cell 1 first each session (loads env vars + BASE).
2. Run Cell 2 to clone/update and install dependencies.
3. Run Cell 3 for OpenEnv/training preflight.
4. Run Cell 4 for the GRPO training run.
5. **Cell 5** — `matplotlib` charts from `grpo_wildfire/log.jsonl` (what the plotter uses: return, grader, loss, KL, parse rate, wall times; curriculum bands by task).
6. **Cell 6** — write the same `submission_artifacts/training_*.png` + `training_summary.md` as `plot_training_curves.py` (for README embeds + `submission_check.py`).

`log.jsonl` is produced by `train_grpo.py` and is *not* always in git; you only have it on the machine where you trained (or a copied `grpo_wildfire/`).

## Training configuration actually used for the submission

The hackathon submission was trained with the **`deadline_v2_a10g`**
configuration in Cell 4 below, which is scheduled for **20 GRPO iterations**:

- `total_iterations=20`
- `task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20))`
- `group_size=4`, `seeds_per_iter=2`, `micro_batch_size=2` — chosen to fill
  the A10G (≈22 GB VRAM) without OOM
- `lora_dropout=0.0`, `warmup_iters=3`, `save_every=6`
- LoRA shape: rank 16, alpha 32, targets `q/k/v/o_proj` (defaults from
  `train_grpo.py`)

**We stopped after iteration 17** due to the hackathon compute budget.
`train_grpo.py` is resume-safe: each iter writes
`grpo_wildfire/latest/`, `optimizer_state.pt`, and `resume_state.json`,
so the iter-17 checkpoint is the one used for the trained-adapter
evaluation in `submission_artifacts/eval_trained.json` (promoted to
`grpo_wildfire/final_adapter/` before eval).

After training finishes, use `notebooks/wildfire_http_eval_hf.ipynb` for
the OpenEnv baseline/trained evaluation and final artifacts.

In [ ]:
# Cell 1 (run first each session): env vars + constants (HF runtime)
import os

GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")
HF_TOKEN = os.environ.get("HF_TOKEN")

if not GITHUB_TOKEN:
    raise RuntimeError("Set GITHUB_TOKEN in the Hugging Face Space/Notebook secrets.")
if not HF_TOKEN:
    raise RuntimeError("Set HF_TOKEN in the Hugging Face Space/Notebook secrets.")

os.environ["HF_TOKEN"] = HF_TOKEN

REPO = "jhawaritvik/Scaler-hackathon"
REPO_URL = f"https://github.com/{REPO}.git"
BASE = "/home/user/app/wildfire-env"

print("Configured BASE:", BASE)
print("Tokens available: GITHUB_TOKEN + HF_TOKEN")

In [ ]:
# Cell 2: clone/update + install deps (safe to re-run)
import os
import glob
import subprocess
import sys

if not os.path.exists(BASE):
    subprocess.run(["git", "clone", f"https://{GITHUB_TOKEN}@github.com/{REPO}.git", BASE], check=True)

os.chdir(BASE)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--rebase", "origin", "main"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".[train]"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "matplotlib", "unsloth", "unsloth-zoo", "bitsandbytes", "xgrammar", "transformers", "trl",
    ],
    check=True,
)

nvidia_lib_dirs = glob.glob("/usr/local/lib/python*/dist-packages/nvidia/*/lib")
if nvidia_lib_dirs:
    os.environ["LD_LIBRARY_PATH"] = ":".join(nvidia_lib_dirs) + ":" + os.environ.get("LD_LIBRARY_PATH", "")
    print("LD_LIBRARY_PATH updated with NVIDIA libs")

print("Install complete. cwd:", os.getcwd())

In [ ]:
# Cell 3: preflight for training stack
import subprocess
import sys

subprocess.run(["openenv", "validate", "."], check=True)
subprocess.run([sys.executable, "smoke_test.py"], check=True)
print("Preflight passed")

In [ ]:
# Cell 4: GRPO training run (resume-safe).
#
# This is the exact `deadline_v2_a10g` configuration used for the hackathon
# submission: 20 GRPO iterations on a single A10G with bigger group/microbatch
# knobs to fill the 23 GB VRAM. We actually stopped after iteration 17 due
# to the compute budget; the resumed iter-17 adapter is what
# `submission_artifacts/eval_trained.json` was generated from.
#
# `train_grpo.py` is resume-safe: every iter writes
# `grpo_wildfire/latest/`, `optimizer_state.pt`, and `resume_state.json`,
# so rerunning this cell loads the latest LoRA + optimizer state and picks
# up from `next_iter`.

import subprocess, sys, os

_HERE = os.path.dirname(os.path.abspath(globals().get("__file__", ".")))
_PROJECT_ROOT = os.path.abspath(os.path.join(_HERE, ".."))
if not os.path.isfile(os.path.join(_PROJECT_ROOT, "train_grpo.py")):
    _PROJECT_ROOT = os.path.expanduser("~/app/wildfire-env")

subprocess.run([
    sys.executable, "-c",
    f"""
import sys
sys.path.insert(0, {_PROJECT_ROOT!r})
from train_grpo import Config, train

train(Config(
    total_iterations=20,
    task_curriculum=(("easy", 0, 5), ("medium", 5, 12), ("hard", 12, 20)),
    group_size=4,           # 2x rollout parallelism per iter
    seeds_per_iter=2,       # more env variance per iter
    micro_batch_size=2,     # fills the 23 GB A10G better
    max_episode_steps=20,
    lora_dropout=0.0,
    warmup_iters=3,
    save_every=6,
    run_name="deadline_v2_a10g",
))
"""
], cwd=_PROJECT_ROOT, check=True)

print("deadline_v2_a10g done (or stopped after iter 17 in our run). "
      "See grpo_wildfire/log.jsonl, grpo_wildfire/latest (resume), "
      "grpo_wildfire/final_adapter (eval).")


In [ ]:
# Cell 5: interactive GRPO curves from the same `log.jsonl` the submission uses
# (every line is one outer iteration, JSON keys from `train_grpo.py` logging)
import json
from pathlib import Path
import matplotlib.pyplot as plt

LOG = Path("grpo_wildfire") / "log.jsonl"
if not LOG.is_file():
    raise FileNotFoundError(
        f"Missing {LOG} — run Cell 4 first, or copy a `grpo_wildfire/` tree from the machine that trained."
    )

def _load_log(path: Path) -> list[dict]:
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line:
            continue
        o = json.loads(line)
        if "iter" in o:
            rows.append(o)
    rows.sort(key=lambda r: int(r["iter"]))
    if not rows:
        raise ValueError("log.jsonl has no iter records")
    return rows

def _task_spans(recs: list[dict]) -> list[tuple[str, int, int]]:
    if not recs:
        return []
    cur, start = str(recs[0]["task_id"]), int(recs[0]["iter"])
    prev = start
    out: list[tuple[str, int, int]] = []
    for r in recs[1:]:
        tid, it = str(r["task_id"]), int(r["iter"])
        if tid != cur:
            out.append((cur, start, prev))
            cur, start = tid, it
        prev = it
    out.append((cur, start, prev))
    return out

TC = {"easy": "#d9f99d", "medium": "#fde68a", "hard": "#fecaca"}
recs = _load_log(LOG)
it = [int(r["iter"]) for r in recs]
spans = _task_spans(recs)

def _shade(ax) -> None:
    for tid, a, b in spans:
        ax.axvspan(a - 0.45, b + 0.45, alpha=0.35, color=TC.get(tid, "#e5e7eb"))

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax in axes:
    _shade(ax)
axes[0].plot(it, [r["mean_return"] for r in recs], "o-", color="#2563eb")
axes[0].set_ylabel("mean_return (rollout group)")
axes[0].set_title("GRPO outer-loop metrics — " + str(LOG))
axes[1].plot(it, [r["mean_grader_score"] for r in recs], "o-", color="#16a34a")
axes[1].set_ylabel("mean_grader_score")
axes[2].plot(it, [r.get("action_parse_success_rate", 0) for r in recs], "o-", color="#ea580c")
axes[2].set_ylabel("action_parse_success_rate")
axes[2].set_xlabel("iteration")
plt.tight_layout()
plt.show()

fig, axr = plt.subplots(2, 2, figsize=(10, 5))
axl = [axr[0, 0], axr[0, 1], axr[1, 0], axr[1, 1]]
for ax in axl:
    _shade(ax)
axl[0].plot(it, [r["policy_loss"] for r in recs], "o-", color="#dc2626")
axl[0].set_ylabel("policy_loss")
axl[1].plot(it, [r["kl_divergence"] for r in recs], "o-", color="#7c3aed")
axl[1].set_ylabel("kl_divergence")
axl[2].plot(it, [r["clip_fraction"] for r in recs], "o-", color="#ea580c")
axl[2].set_ylabel("clip_fraction")
axl[3].plot(it, [r.get("rollout_seconds", 0) for r in recs], "o-", color="#0ea5e9", label="rollout_s")
axl[3].plot(it, [r.get("update_seconds", 0) for r in recs], "s--", color="#64748b", label="update_s")
axl[3].set_ylabel("wall clock (s)")
axl[3].legend(loc="best", fontsize=8)
axl[2].set_xlabel("iteration")
axl[3].set_xlabel("iteration")
plt.tight_layout()
plt.show()

print(f"Loaded {len(recs)} iters; curriculum spans: {spans}")

In [ ]:
# Cell 6: same PNGs / SVGs / `training_summary.md` as the old `python plot_training_curves.py`
# (PIL, no extra matplotlib dependency for the file writer — use this for git + README embeds)
from pathlib import Path
from plot_training_curves import generate_training_plots

_log = Path("grpo_wildfire") / "log.jsonl"
out = Path("submission_artifacts")
out.mkdir(parents=True, exist_ok=True)
generate_training_plots(_log, out)

# Stop Here
#
# After training, run **Cells 5–6** to inspect `log.jsonl` in the notebook and
# to export `submission_artifacts/training_*.png` (required for the README
# and `submission_check.py`).
#
# For the hackathon we ran 17 of 20 scheduled iters; the iter-17 adapter under
# `grpo_wildfire/latest/` was used for the trained OpenEnv eval. Use
# `notebooks/wildfire_http_eval_hf.ipynb` for eval JSONs and (optionally) more charts.
